# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [ ]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [4]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")

df=pd.read_csv(filepath)

feat_col = "Desafío País"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Desafío País"].notna()]

#Eliminar duplicados
df = df.drop_duplicates("Código VRID")

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Desafío País"])
savepath = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)


Test size: 247
Fold 0 - Val size: 330
Archivo guardado exitosamente en /tmp/final_project/dataSplits/desafios/train_test_ids_3folds.json


# 1) TF-IDF

In [5]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [ ]:
"""
path2 = "/tmp/desafios/new_data"
filepath=os.path.join(path2, "new_data_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path2, "data_translated.csv")
df = pd.read_csv(filepath)
"""

In [6]:
from preprocess.preprocess import one_hot_codification

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

# Crear labels 
df = one_hot_codification(df, "Desafío País", comb_desafios)


In [7]:
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline
from utils.mlflow import mlflow_ckeckpoint


warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)
#Rutas
split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
savepath = os.path.join(path, "output/desafios/TF_IDF")
model_paths = ["1", "2", "3", "4"]

#Mlfow
exp_info = {
    'exp_name': "final_project_desafios",
}
extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

for idx, des in enumerate(comb_desafios):
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    #Creacion de vectores TFID
    X_train, X_test, vectorizer = gen_TFID_vectors(X_train, X_test, return_vectorizer=True)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    results_test, preds_test = {}, {}
    for name, model in models_dicc.items():
        print(name)
        results, preds = eval_model(model, X_test, y_test, lang_es)
        print(results)
        results_test[name] = results
        preds_test[name] = preds

    models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
    save_models_and_metrics(savepath, results_val, models_dicc_pipeline, df_test, y_test, results_test, preds_test, save_preds=True, mode_classification="binary")
    
    #Mlflow para creación de reportes
    extra_parms = {
        "vectorization_model": "TF-IDF",
        "n_iter": n_iter,
        "sample_weight_On": sample_weight_On,
        "scoring": scoring,
        "cols": cols,
        "desafio": str(des)
    }

    mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, 
                      lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, 
                      mode="server", mode_classification="binary")
    

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


(987, 18519) (247, 18519)
📊 train: Counter({0: 887, 1: 100})
📊 test: Counter({0: 222, 1: 25})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.72, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.7, 'std_test_score': 0.04}
XGBClassifier: {'mean_test_score': 0.72, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.71, 'std_test_score': 0.02}
LogisticRegression
{'accuracy': 0.9352226720647774, 'f1_macro': 0.8004040404040403, 'cm': array([[217,   5],
       [ 11,  14]]), 'precision': 0.7368421052631579, 'recall': 0.56, 'f1_es': 0.9094128890804489, 'f1_en': 0.9641501650165016, 'cm_es': array([[123,   2],
       [ 10,  11]]), 'cm_en': array([[94,  3],
       [ 1,  3]])}
RandomForestClassifier
{'accuracy': 0.9109311740890689, 'f1_macro': 0.7552252252252252, 'cm': array([[211,  11],
       [ 11,  14]]), 'precision': 0.56, 'recall': 0.56, 'f1_es': 0.871114

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


2025/09/29 15:21:29 INFO mlflow.tracking.fluent: Experiment with name 'final_project_desafios' does not exist. Creating a new experiment.


...

🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/26/runs/f6f6aafb2a4f482a8df48fb7773b3d58
🧪 View experiment at: http://mlflow-server:5000/#/experiments/26
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/29 15:35:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/26/runs/3d5b2e441e1b4779b0f695b178abd4b6
🧪 View experiment at: http://mlflow-server:5000/#/experiments/26
📝 Registrando modelo en MLflow: SVC


2025/09/29 15:35:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/26/runs/12a9a984c9124f63a9fd5df09e48729c
🧪 View experiment at: http://mlflow-server:5000/#/experiments/26


### Inference

In [8]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/desafios/TF_IDF/models"
model_path = os.path.join(model_path)

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Mejor modelo: LogisticRegression
Prediction: [1]


# 2) SPECTER

In [9]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [10]:
from preprocess.preprocess import one_hot_codification

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

# Crear labels 
df = one_hot_codification(df, "Desafío País", comb_desafios)


In [11]:
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline
from models.specter import BERT_vectorizer

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
#Guardado de resultados
savepath = os.path.join(path, "output/desafios/SPECTER")
model_paths = ["1", "2", "3", "4"]

#Mlfow
exp_info = {
    'exp_name': "final_project_desafios",
}
extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

for des in comb_desafios:
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    # Calcular embeddings
    # Parámetros para cargar modelo
    BASE_MODEL = "allenai/specter2_base"
    ADAPTER_NAME="allenai/specter2_classification"
    X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
    X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    results_test, preds_test = {}, {}
    for name, model in models_dicc.items():
        print(name)
        results, preds = eval_model(model, X_test, y_test, lang_es)
        print(results)
        results_test[name] = results
        preds_test[name] = preds
    
    vectorizer = BERT_vectorizer(BASE_MODEL, ADAPTER_NAME)
    models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
    save_models_and_metrics(savepath, results_val, models_dicc_pipeline, df_test, y_test, results_test, 
                            preds_test, save_preds=True, mode_classification="binary")
    
    #Mlflow para creación de reportes
    extra_parms = {
        "vectorization_model": "SPECTER",
        "n_iter": n_iter,
        "sample_weight_On": sample_weight_On,
        "scoring": scoring,
        "cols": cols,
        "desafio": str(des)
    }

    mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, 
                      lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, 
                      mode="server", mode_classification="binary")

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/26/runs/7e9a669fbd224f398aec3a5d335cfec4
🧪 View experiment at: http://mlflow-server:5000/#/experiments/26
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/29 16:12:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/26/runs/31cd6cf433154f7b94cc8e8aa542157c
🧪 View experiment at: http://mlflow-server:5000/#/experiments/26
📝 Registrando modelo en MLflow: SVC


2025/09/29 16:12:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/26/runs/1e99c6c166204ef683260a703fff4a51
🧪 View experiment at: http://mlflow-server:5000/#/experiments/26


### Inference

In [30]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/desafios/SPECTER/models"
model_path = os.path.join(model_path)

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Mejor modelo: LogisticRegression


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
Prediction: [0]
